In [1]:
from rrt_star import *
import numpy as np
import matplotlib.pyplot as plt
import roboticstoolbox as rtb
import spatialmath as sm # provides objects for representing transformations
from swift import Swift # lightweight browser-based simulator which comes with the toolbox
import spatialgeometry as sg # utility package for dealing with geometric objects

In [2]:
robot = rtb.models.URDF.wx200()
print(robot)

moving_link_idxs = [0,1,2,3,4]
colliding_link_idxs  = [0,1,2,3,4,5,6,7,8,9,10,11,12]

def pad(q):
	return np.hstack([q, [0]*(len(robot.q)-len(moving_link_idxs))])

ERobot: wx200 (by Interbotix), 8 joints (RRRRRRPP), 4 branches, dynamics, geometry, collision
┌──────┬─────────────────────┬───────┬───────────────────┬───────────────────────────────────────────┐
│ link │        link         │ joint │      parent       │            ETS: parent to link            │
├──────┼─────────────────────┼───────┼───────────────────┼───────────────────────────────────────────┤
│    0 │ /base_link          │       │ BASE              │ SE3()                                     │
│    1 │ /shoulder_link      │     0 │ /base_link        │ SE3(0, 0, 0.0716) ⊕ Rz(q0)                │
│    2 │ /upper_arm_link     │     1 │ /shoulder_link    │ SE3(0, 0, 0.03865) ⊕ Ry(q1)               │
│    3 │ /forearm_link       │     2 │ /upper_arm_link   │ SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q2) │
│    4 │ /wrist_link         │     3 │ /forearm_link     │ SE3(0.2, 0, 0) ⊕ Ry(q3)                   │
│    5 │ /gripper_link       │     4 │ /wrist_link       │ SE3(0.065, 0, 0; -180°,

In [3]:
sp = sg.Sphere(0.2)
sp.T = sp.T * sm.SE3.Trans(0,0.4,0.6)
obstacles = [sp]

In [4]:
# robot.collided(robot.q, obstacles[0])

In [46]:
sp = sg.Sphere(0.2)
sp.T = sp.T * sm.SE3.Trans(0,0.4,0.6)

sp2 = sg.Sphere(0.2)
sp2.T = sp2.T * sm.SE3.Trans(0.2,0.4,0.4)

obstacles = [sp,sp2]

def has_collision(q):
	robot.q = pad(q)
	links = [robot.links[i] for i in colliding_link_idxs]
	n_links = len(links)

	# doesn't work since robot is in collision with itself
	# for obstacle in obstacles:
	# 	if robot.iscollided(robot.q, obstacle):
	# 		return True
	for i1 in range(n_links):
		link1 = links[i1]

		# check if this link collides with any obstacles
		for obstacle in obstacles:
			for collider in link1.collision:
				if collider.iscollided(obstacle):
					print(f'found a collision between (link {link1}) and ({obstacle})')
					return True

		# check if this link collides with any other links
		for i2 in range(n_links):
			if abs(i1-i2)<2:
				continue # don't check adjacent links
			link2 = links[i2]
			for collider1 in link1.collision:
				if collider1.stype == 'mesh':
					continue
				for collider2 in link2.collision:
					if collider2.stype == 'mesh':
						continue
					if collider1.iscollided(collider2):
						return True
	return False

has_collision(robot.q[moving_link_idxs])

False

In [47]:
goal_ee_T = robot.fkine((1/4*robot.qlim[0]+3/4*robot.qlim[1]), end='/ee_arm_link')
goal_estimate = robot.ikine_QP(Tep=goal_ee_T, end='/ee_arm_link').q

def goal_test(q):
	ee_T = robot.fkine(pad(q), end='/ee_arm_link')
	return np.sum(np.abs(ee_T-goal_ee_T)) < 0.01

In [48]:
start_config = np.zeros(len(moving_link_idxs)) # Initial configuration

planner = RRTStar(start=start_config, goal=goal_estimate, 
					minCoords=robot.qlim[0][moving_link_idxs], maxCoords=robot.qlim[1][moving_link_idxs], 
					goal_test=goal_test, step_size=0.2, search_radius=0.5, collisionChecker=has_collision,
					seed=10)
path = planner.run_rrt_star()

if path:
	print(f'Found path of length {len(path)}:')
	for config in path:
		print(config, has_collision(config))
else:
	print("Path not found.")

Goal reached in 269 iterations.
Found path of length 15:
[0. 0. 0. 0. 0.] False
[ 0.22249837  0.06221866  0.20122416 -0.02922913  0.02984872] False
[ 0.6278768  -0.11373053  0.39911415  0.03707043  0.07540226] False
[1.04112939 0.00829148 0.215191   0.01371814 0.02048944] False
[1.27529136 0.05799683 0.28911356 0.14754615 0.21661347] False
[1.39711461 0.19208552 0.37150248 0.15132264 0.23603316] False
[1.5057136  0.45289804 0.49102873 0.16413477 0.34066889] False
[1.51915251 0.63463954 0.73412222 0.22047831 0.59467751] False
[1.53259142 0.81638104 0.97721572 0.27682186 0.84868613] False
[1.53931088 0.90725179 1.09876247 0.30499363 0.97569044] False
[1.54603033 0.99812254 1.22030922 0.3331654  1.10269475] False
[1.55274979 1.08899329 1.34185597 0.36133717 1.22969905] False
[1.55946925 1.17986404 1.46340272 0.38950894 1.35670336] False
[1.57079633 1.33304615 1.6682958  0.4369985  1.57079633] False
[1.57079633 1.33304615 1.6682958  0.4369985  1.57079633] False


In [54]:
env = Swift()
robot.q = pad(start_config)
env.launch(realtime=True, browser="notebook")
env.add(robot)
env.add(ee_axes)
env.add(goal_axes)

for step in path:
	robot.q = pad(step)
	print(has_collision(step))
	ee_axes.T = robot.fkine(robot.q, end='/ee_arm_link').A
	env.step(dt)

False
False
False
found a collision between (link Link("/gripper_bar_link", SE3(), parent="/ee_arm_link", m=0.0342, r=[0.00969, 8.18e-07, 0.00496], I=[7.41e-06, 2.84e-05, 2.86e-05, -8e-10, -6e-10, -1.39e-06], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
True
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q), parent="/upper_arm_link", qlim=[-1.62, 1.88], m=0.284, r=[0.121, -0.000124, 0], I=[0.00119, 6.83e-05, 0.00121, -2.43e-06, 0, 0], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
True
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q), parent="/upper_arm_link", qlim=[-1.62, 1.88], m=0.284, r=[0.121, -0.000124, 0], I=[0.00119, 6.83e-05, 0.00121, -2.43e-06, 0, 0], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
True
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q), parent="/upper_arm_l

In [51]:
ee_axes = sg.Axes(0.1)
goal_axes = sg.Axes(0.1)
goal_axes.T = goal_ee_T

# Make a new environment and add our robot
env = Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
env.add(ee_axes)
env.add(goal_axes)

for obstacle in obstacles:
	env.add(obstacle)

# Change the robot configuration to the ready position
robot.q = pad(start_config)

# Step the sim to view the robot in this configuration
env.step(0.01)

dt = 0.5
time = 0

for step_i in range(len(path)):
	# robot.q = pad(path[step_i])

	if has_collision(path[step_i]):
		print('found collision at step', step_i)
	
	# Update the ee axes
	ee_axes.T = robot.fkine(robot.q, end='/ee_arm_link').A
	
	# Step the simulator by dt seconds
	env.step(dt)
	time += dt

found a collision between (link Link("/gripper_bar_link", SE3(), parent="/ee_arm_link", m=0.0342, r=[0.00969, 8.18e-07, 0.00496], I=[7.41e-06, 2.84e-05, 2.86e-05, -8e-10, -6e-10, -1.39e-06], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
found collision at step 3
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q), parent="/upper_arm_link", qlim=[-1.62, 1.88], m=0.284, r=[0.121, -0.000124, 0], I=[0.00119, 6.83e-05, 0.00121, -2.43e-06, 0, 0], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
found collision at step 4
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2; 180°, -0°, 0°) ⊕ Ry(q), parent="/upper_arm_link", qlim=[-1.62, 1.88], m=0.284, r=[0.121, -0.000124, 0], I=[0.00119, 6.83e-05, 0.00121, -2.43e-06, 0, 0], Jm=0, B=0, Tc=[0, 0], G=0)) and (stype: sphere 
 pose: [0.2 0.4 0.4])
found collision at step 5
found a collision between (link Link("/forearm_link", SE3(0.05, 0, 0.2;